# Brownian Motion and Brownian Bridge

Brownian motion — also called the **Wiener process** — is one of the most fundamental objects in probability theory, stochastic analysis, and mathematical physics. It was first observed by the botanist Robert Brown (1827) as the erratic displacement of pollen grains suspended in water, and given a rigorous mathematical foundation by Norbert Wiener (1923). Louis Bachelier (1900) had already applied it to price modeling five years before Einstein's famous 1905 paper on diffusion.

A **standard Brownian motion** $\{B_t\}_{t \geq 0}$ is a real-valued stochastic process characterized by four properties:

1. **Initial condition**: $B_0 = 0$ almost surely.
2. **Independent increments**: for $0 \leq s < t$, $B_t - B_s$ is independent of $\sigma(B_r : r \leq s)$.
3. **Stationary Gaussian increments**: $B_t - B_s \sim \mathcal{N}(0,\, t - s)$.
4. **Continuous paths**: $t \mapsto B_t$ is almost surely continuous.

The covariance structure is entirely determined by
$$
\mathrm{Cov}(B_s,\, B_t) = \mathbb{E}[B_s B_t] = \min(s, t).
$$

This notebook explores how Brownian motion arises as the scaling limit of a random walk (**Donsker's theorem**), visualizes properties of 1D and 2D paths, examines the famous **quadratic variation**, and constructs the **Brownian bridge** — a Brownian motion pinned to fixed endpoints. Interactive controls let you explore the parameter space directly.

## Environment

Standard scientific Python stack; `ipywidgets` for interactive sliders.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import norm
from ipywidgets import interact, IntSlider, FloatSlider

plt.rcParams["figure.dpi"] = 120
rng = np.random.default_rng(0)

## From random walk to Brownian motion

Brownian motion is the scaling limit of a simple random walk. Given $n$ i.i.d. increments $\xi_k \sim \mathcal{N}(0, 1/n)$, the cumulative sum
$$
B^{(n)}_t = \sum_{k=1}^{\lfloor nt \rfloor} \xi_k, \qquad t \in [0, 1],
$$
converges in distribution to a standard Brownian motion $B$ as $n \to \infty$ (**Donsker's invariance principle**).

Below, three resolutions $n \in \{16, 128, 2048\}$ illustrate this convergence for five independent trajectories.

In [ ]:
def make_bm(n, k, seed=None):
    """k independent Brownian paths on [0,1] with n steps."""
    r = np.random.default_rng(seed)
    inc = r.normal(0, 1 / np.sqrt(n), (n, k))
    return np.vstack([np.zeros((1, k)), np.cumsum(inc, axis=0)])

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, n in zip(axes, [16, 128, 2048]):
    paths = make_bm(n, 6, seed=7)
    t = np.linspace(0, 1, n + 1)
    for j in range(paths.shape[1]):
        s = j / (paths.shape[1] - 1)
        ax.plot(t, paths[:, j], lw=1.5, color=(s, 0.15, 1 - s))
    ax.set_title(f"$n = {n}$ steps", fontsize=11)
    ax.axhline(0, color="k", lw=0.5, ls="--")
    ax.set_xlim(0, 1)
    ax.set_xlabel("$t$")
    ax.set_ylabel("$B_t$" if ax is axes[0] else "")
fig.suptitle("Donsker convergence: random walk $\\to$ Brownian motion", y=1.02)
plt.tight_layout()
plt.show()

## Many trajectories and the Gaussian envelope

At each fixed time $t$, $B_t \sim \mathcal{N}(0, t)$. The standard deviation envelope $\pm\sqrt{t}$ therefore grows as a square root. The figure below overlays 40 paths and the $\pm 2\sqrt{t}$ band that should capture about 95 % of paths.

In [ ]:
n, k = 2000, 40
paths = make_bm(n, k, seed=42)
t = np.linspace(0, 1, n + 1)

fig, ax = plt.subplots(figsize=(9, 4))
for j in range(k):
    s = j / (k - 1)
    ax.plot(t, paths[:, j], lw=0.8, alpha=0.55, color=(s, 0.1, 1 - s))
ax.fill_between(t, -2 * np.sqrt(t), 2 * np.sqrt(t),
                alpha=0.15, color="gold", label=r"$\pm 2\sqrt{t}$ band (95 %)")
ax.plot(t, 2 * np.sqrt(t), color="goldenrod", lw=1.8, ls="--")
ax.plot(t, -2 * np.sqrt(t), color="goldenrod", lw=1.8, ls="--")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("$t$"); ax.set_ylabel("$B_t$")
ax.set_title(f"{k} independent Brownian paths with $\\pm 2\\sqrt{{t}}$ envelope")
ax.legend()
plt.tight_layout()
plt.show()

## Marginal distribution at fixed time

For a fixed time $t^*$, $B_{t^*} \sim \mathcal{N}(0, t^*)$. We verify this by drawing many sample paths and histogramming the terminal values $B_{t^*}$, then overlaying the theoretical density
$$
p_{t^*}(x) = \frac{1}{\sqrt{2\pi t^*}} \exp\!\left(-\frac{x^2}{2t^*}\right).
$$

In [ ]:
def show_marginal(t_star=0.5):
    n_mc = 8000
    inc = rng.normal(0, 1 / np.sqrt(1000), (1000, n_mc))
    b_all = np.cumsum(inc, axis=0)
    idx = min(int(t_star * 1000), 999)
    samples = b_all[idx]

    xg = np.linspace(-4 * np.sqrt(t_star), 4 * np.sqrt(t_star), 300)
    density = norm.pdf(xg, 0, np.sqrt(t_star))

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.hist(samples, bins=60, density=True, alpha=0.55, color="steelblue",
            label=f"empirical ($n=8000$)")
    ax.plot(xg, density, "r-", lw=2,
            label=r"$\mathcal{N}(0,\,t^*)$")
    ax.set_title(f"$B_{{t^*}}$ at $t^* = {t_star:.2f}$")
    ax.set_xlabel("$x$"); ax.set_ylabel("density")
    ax.legend()
    plt.tight_layout(); plt.show()

interact(show_marginal,
         t_star=FloatSlider(value=0.5, min=0.05, max=1.0, step=0.05,
                            description="$t^*$"));

## Planar (2D) Brownian motion

A **planar Brownian motion** is the complex-valued process
$$
Z_t = B_t^{(1)} + i\, B_t^{(2)},
$$
where $B^{(1)}$ and $B^{(2)}$ are independent standard Brownian motions. The density of $Z_t$ is a 2D isotropic Gaussian:
$$
p_t(x, y) = \frac{1}{2\pi t} \exp\!\left(-\frac{x^2 + y^2}{2t}\right).
$$

Paths are colored by time from blue (start) to red (end), and each trajectory begins from the origin.

In [ ]:
def make_bm2d(n, k, seed=0):
    r = np.random.default_rng(seed)
    inc = (r.normal(0, 1 / np.sqrt(n), (n, k))
           + 1j * r.normal(0, 1 / np.sqrt(n), (n, k)))
    return np.vstack([np.zeros((1, k)), np.cumsum(inc, axis=0)])

n2d, k2d = 3000, 6
Z = make_bm2d(n2d, k2d, seed=5)

fig, axes = plt.subplots(1, k2d, figsize=(12, 2.4))
cmap = cm.coolwarm
for j, ax in enumerate(axes):
    seg_x = Z[:, j].real
    seg_y = Z[:, j].imag
    colors = cmap(np.linspace(0, 1, n2d))
    for i in range(0, n2d, max(1, n2d // 400)):
        ax.plot(seg_x[i:i+2], seg_y[i:i+2], lw=0.9,
                color=colors[i], alpha=0.85)
    ax.plot(0, 0, "ko", ms=3)
    ax.plot(seg_x[-1], seg_y[-1], "r.", ms=6)
    lim = 1.8
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal"); ax.axis("off")
fig.suptitle("Six independent planar Brownian paths  (blue=start, red=end)",
             y=1.04)
plt.tight_layout()
plt.show()

## Quadratic variation

A hallmark of Brownian motion is its **quadratic variation**. For any partition $0 = t_0 < t_1 < \cdots < t_n = T$ with mesh $\|\Pi\| \to 0$,
$$
\sum_{k=0}^{n-1} \bigl(B_{t_{k+1}} - B_{t_k}\bigr)^2 \xrightarrow{\;L^2\;} T.
$$

This is often written $d[B]_t = dt$, reflecting that Brownian paths oscillate too wildly to have finite first-order variation ($p$-variation with $p < 2$ is infinite), but their squared increments add up to exactly the elapsed time.

We illustrate this convergence by computing the sum of squared increments on increasingly fine partitions.

In [ ]:
n_fine = 65536
single = make_bm(n_fine, 1, seed=3)[:, 0]

n_vals = np.unique(np.round(np.logspace(1, np.log10(n_fine), 60)).astype(int))
qv = []
for n_sub in n_vals:
    idx = np.round(np.linspace(0, n_fine, n_sub + 1)).astype(int)
    inc = np.diff(single[idx])
    qv.append(np.sum(inc ** 2))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogx(n_vals, qv, "o-", ms=3, lw=1.3, color="steelblue",
            label="empirical $\\sum (\\Delta B)^2$")
ax.axhline(1.0, color="crimson", lw=1.8, ls="--", label="$T = 1$")
ax.set_xlabel("partition size $n$")
ax.set_ylabel("quadratic variation")
ax.set_title("Quadratic variation converges to $T$")
ax.legend()
plt.tight_layout()
plt.show()

## Brownian bridge

A **Brownian bridge** from $a$ at time $0$ to $b$ at time $1$ is a Brownian motion conditioned to satisfy the endpoint constraint $\beta_1 = b$. It can be constructed explicitly from a free Brownian motion $B$:
$$
\beta_t = (1 - t)\,a + t\,b + \sigma\,\bigl(B_t - t\, B_1\bigr), \qquad t \in [0, 1].
$$

The term $B_t - t B_1$ is itself a Gaussian bridge pinned to $0$ at both endpoints (when $a = b = 0$). Its covariance is
$$
\mathrm{Cov}(\beta_s, \beta_t) = \sigma^2 s(1-t), \qquad s \leq t.
$$

The maximum variance occurs at $t = 1/2$ and equals $\sigma^2/4$.

In [ ]:
def brownian_bridge(n, k, a=0.0, b=0.0, sigma=1.0, seed=0):
    """Construct k Brownian bridges from a to b with noise scale sigma."""
    r = np.random.default_rng(seed)
    inc = r.normal(0, 1 / np.sqrt(n), (n, k))
    B = np.vstack([np.zeros((1, k)), np.cumsum(inc, axis=0)])  # free BM
    t = np.linspace(0, 1, n + 1)[:, None]
    pinned = B - t * B[-1]                           # zero-bridge
    return (1 - t) * a + t * b + sigma * pinned      # shifted bridge


n_br = 800
sigma_vals = [0.05, 0.2, 0.5, 1.0]
t = np.linspace(0, 1, n_br + 1)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, sigma in zip(axes, sigma_vals):
    bridges = brownian_bridge(n_br, 8, a=0.0, b=0.0, sigma=sigma, seed=1)
    env = 2 * sigma * np.sqrt(t * (1 - t))          # ± 2 std envelope
    for j in range(8):
        s = j / 7
        ax.plot(t, bridges[:, j], lw=1.0, alpha=0.7, color=(s, 0.1, 1 - s))
    ax.fill_between(t, -env, env, alpha=0.12, color="gold")
    ax.plot(t, env, "--", color="goldenrod", lw=1.2)
    ax.plot(t, -env, "--", color="goldenrod", lw=1.2)
    ax.plot([0, 1], [0, 0], "ko", ms=4)
    ax.set_title(f"$\\sigma = {sigma}$")
    ax.set_xlim(0, 1); ax.set_xlabel("$t$")
    ax.set_ylabel("$\\beta_t$" if ax is axes[0] else "")
fig.suptitle("Brownian bridges $(a=0, b=0)$ for varying $\\sigma$", y=1.02)
plt.tight_layout()
plt.show()

## Interactive bridge: endpoint and noise exploration

The slider lets you control the arrival point $b$, the noise amplitude $\sigma$, and the number of independent sample paths simultaneously rendered.

In [ ]:
def show_bridge(b=0.0, sigma=0.5, k=10):
    n_b = 600
    bridges = brownian_bridge(n_b, k, a=0.0, b=b, sigma=sigma, seed=2)
    t_b = np.linspace(0, 1, n_b + 1)
    mean = (1 - t_b) * 0 + t_b * b
    env = 2 * sigma * np.sqrt(t_b * (1 - t_b))

    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    for j in range(k):
        s = j / max(k - 1, 1)
        ax.plot(t_b, bridges[:, j], lw=1.0, alpha=0.6, color=(s, 0.1, 1 - s))
    ax.plot(t_b, mean, "k--", lw=1.5, label="mean $(1-t)a + tb$")
    ax.fill_between(t_b, mean - env, mean + env,
                    alpha=0.15, color="gold", label=r"$\pm 2\sigma\sqrt{t(1-t)}$")
    ax.plot([0, 1], [0, b], "ko", ms=5)
    ax.set_xlabel("$t$"); ax.set_ylabel("$\\beta_t$")
    ax.set_title(f"Brownian bridge: $a=0$, $b={b:.2f}$, $\\sigma={sigma:.2f}$, $k={k}$")
    ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()

interact(
    show_bridge,
    b=FloatSlider(value=0.0, min=-1.5, max=1.5, step=0.1, description="$b$"),
    sigma=FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05, description="$\\sigma$"),
    k=IntSlider(value=10, min=2, max=30, step=1, description="paths"),
);

## Covariance structure

The covariance function of a Brownian motion is $K(s,t) = \min(s,t)$, while for the zero Brownian bridge it is $K(s,t) = s(1-t)$ for $s \leq t$. Both are **positive-definite kernels** and entirely determine the distribution of the respective process.

We estimate the covariance matrix empirically from 5000 sample paths and compare to the theoretical matrix.

In [ ]:
m_grid = 40          # coarse grid for covariance display
n_mc   = 5000

# Brownian motion on m_grid time points
bm_mc  = make_bm(m_grid, n_mc, seed=99)[1:]   # drop t=0
t_grid = np.linspace(1 / m_grid, 1, m_grid)

cov_emp_bm = np.cov(bm_mc)
cov_th_bm  = np.minimum(t_grid[:, None], t_grid[None, :])

# Brownian bridge
bb_mc  = brownian_bridge(m_grid, n_mc, seed=99)[1:]
cov_emp_bb = np.cov(bb_mc)
s, t2  = t_grid[:, None], t_grid[None, :]
cov_th_bb  = np.where(s <= t2, s * (1 - t2), t2 * (1 - s))

vmax = 0.5
fig, axes = plt.subplots(2, 2, figsize=(8, 7.5))
titles = ["BM empirical", "BM theoretical $\\min(s,t)$",
          "Bridge empirical", "Bridge theoretical $s(1-t)$"]
for ax, C, title in zip(axes.ravel(),
                         [cov_emp_bm, cov_th_bm, cov_emp_bb, cov_th_bb],
                         titles):
    im = ax.imshow(C, origin="lower", vmin=0, vmax=vmax,
                   extent=[0, 1, 0, 1], cmap="viridis")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("$t$"); ax.set_ylabel("$s$")
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Empirical vs theoretical covariance", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## Bibliographical resources

- Wiener, N. (1923). *Differential space*. Journal of Mathematics and Physics.
- Bachelier, L. (1900). *Théorie de la spéculation*. Annales de l'École Normale Supérieure.
- Lévy, P. (1948). *Processus stochastiques et mouvement brownien*. Gauthier-Villars.
- Donsker, M. D. (1951). An invariance principle for certain probability limit theorems. *Memoirs of the AMS*.
- Revuz, D. and Yor, M. (1999). *Continuous Martingales and Brownian Motion* (3rd ed.). Springer.
- Mörters, P. and Peres, Y. (2010). *Brownian Motion*. Cambridge University Press.